In [ ]:
import torch
import datasets
import evaluate
import numpy as np
import os
import pandas as pd
import librosa
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor
from datasets import Audio
from transformers import WhisperForConditionalGeneration
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer

2025-07-28 19:45:32.224930: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753731932.448895      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753731932.517708      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
import os
HF_TOKEN = "some_token"

In [4]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [5]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_csv("/kaggle/input/the-uyghur-voice-cup/train.csv")

df = df.rename(columns={"transcription": "sentence"})

df["filepath"] = df["filepath"].apply(lambda x: f"/kaggle/input/the-uyghur-voice-cup/wavs/{os.path.basename(x)}")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_test = df.iloc[:100]
df_train = df.iloc[100:]

ds_train = Dataset.from_pandas(df_train)
ds_test = Dataset.from_pandas(df_test)

ds = DatasetDict({
    "train": ds_train,
    "test": ds_test
})

print(ds)

DatasetDict({
    train: Dataset({
        features: ['ID', 'filepath', 'sentence'],
        num_rows: 7474
    })
    test: Dataset({
        features: ['ID', 'filepath', 'sentence'],
        num_rows: 100
    })
})


In [6]:
def load_audio(batch):
    audio, _ = librosa.load(batch["filepath"], sr=16000)
    batch["audio"] = {"array": audio, "sampling_rate": 16000}
    return batch

ds = ds.map(load_audio)

Map:   0%|          | 0/7474 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", task="transcribe")

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [8]:
def prepare(example):
    audio = example["audio"]
    # print(audio["array"])
    inputs = processor(audio["array"], sampling_rate=16000, return_attention_mask=True)
    example["input_features"] = inputs.input_features[0]
    example["input_length"] = len(example["input_features"])
    example["labels"] = tokenizer(example["sentence"]).input_ids
    return example

In [9]:
ds = ds.map(prepare, remove_columns=ds["train"].column_names)

Map:   0%|          | 0/7474 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [11]:
model.config.task = "transcribe"
model.config.forced_decoder_ids = None

In [12]:
@dataclass
class DataCollatorWhisperWithPadding:
  processor: WhisperProcessor
  padding: Union[bool,str]=True

  def __call__(self, features: List[Dict[str,Any]]) -> Dict[str, torch.Tensor]:
    input_features=[{"input_features":f["input_features"]} for f in features]

    batch=self.processor.feature_extractor.pad(
        input_features,
        padding=self.padding,
        return_tensors="pt"
    )
    if "labels" in features[0]:
            label_features = [{"input_ids": f["labels"]} for f in features]
            labels_batch = self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt"
            )
            labels = labels_batch["input_ids"].masked_fill(
                labels_batch.attention_mask.ne(1), -100
            )
            batch["labels"] = labels
    return batch

In [13]:
data_collator = DataCollatorWhisperWithPadding(
    processor=processor,
    padding = True,
)

In [14]:
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


In [15]:
training_args = Seq2SeqTrainingArguments(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  
    learning_rate=5e-5,
    warmup_steps=200,
    weight_decay=0.01,
    num_train_epochs=3,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=200,
    save_total_limit=2,
    eval_steps=100,
    logging_steps=25,
    logging_strategy="steps",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    remove_unused_columns=False
)

In [16]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=data_collator,
    tokenizer=processor,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_19/887825025.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [17]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Cer
100,1.556400,1.436943,0.272276
200,1.030800,1.012544,0.224678
300,0.892500,0.841732,0.233206
400,0.740100,0.736547,0.160198
500,0.688000,0.661421,0.143839
600,0.634500,0.625297,0.164810
700,0.568300,0.566538,0.118778
800,0.483200,0.518547,0.132788
900,0.452200,0.487434,0.112948
1000,0.299000,0.460868,0.113209


Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3464: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183

TrainOutput(global_step=2805, training_loss=0.42966846521950447, metrics={'train_runtime': 8445.9501, 'train_samples_per_second': 2.655, 'train_steps_per_second': 0.332, 'total_flos': 5.5200464216064e+17, 'train_loss': 0.42966846521950447, 'epoch': 3.0})

In [18]:
import pandas as pd
import torch
import librosa
from tqdm import tqdm
import os

test_df = pd.read_csv("/kaggle/input/the-uyghur-voice-cup/test.csv")
test_df["filepath"] = test_df["filepath"].apply(
    lambda x: os.path.join("/kaggle/input/the-uyghur-voice-cup/wavs", os.path.basename(x))
)

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ids, preds = [], []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):

    audio, _ = librosa.load(row["filepath"], sr=16000)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

    ids.append(row["ID"])
    preds.append(transcription)

submission = pd.DataFrame({"ID": ids, "transcription": preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved.")


100%|██████████| 1894/1894 [10:47<00:00,  2.93it/s]

submission.csv saved.
